Reference

1. [Detailed Explanation and Python Implementation of Q-Learning Algorithm in OpenAI Gym (Cart-Pole)](https://www.youtube.com/watch?v=KMjQmG5Uzis)


### Basic Imports

In [ ]:
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
import pandas as pd

import numpy as np

%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
env = gym.make('MountainCar-v0', render_mode='human')

In [ ]:
env.reset(seed=42)
for i in range(100):
    env.render()
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    print(i, observation, reward, terminated, truncated, info)
    if terminated:
        break
    

In [ ]:
nbins_pos = 3
nbins_vel = 3

pos_bins = np.linspace(env.observation_space.low[0], env.observation_space.high[0], nbins_pos)
vel_bins = np.linspace(env.observation_space.low[1], env.observation_space.high[1], nbins_vel)

In [ ]:
pos_bins, vel_bins

In [ ]:
# Create a Q-table in pandas with multi-index
# index in the form of (pos, vel)
# columns are the actions
n_actions = env.action_space.n


# Create labels for bins with desired format
pos_labels = [f'{pos1:.2f} < pos < {pos2:.2f}' for pos1, pos2 in zip(pos_bins[:-1], pos_bins[1:])]
vel_labels = [f'{vel1:.2f} < vel < {vel2:.2f}' for vel1, vel2 in zip(vel_bins[:-1], vel_bins[1:])]

# Create MultiIndex for the Q-table
index = pd.MultiIndex.from_product([pos_labels, vel_labels], names=['pos', 'vel'])



columns = range(n_actions)

# Create Q-table with MultiIndex
q_table = pd.DataFrame(0, index=index, columns=columns)

In [ ]:
q_table

In [ ]:
pos_bins

In [ ]:
# adding -inft to the first and +inft to the last bin
pos_bins = np.concatenate([[-np.inf], pos_bins, [np.inf]])
vel_bins = np.concatenate([[-np.inf], vel_bins, [np.inf]])

# Create labels for bins with desired format
pos_labels = [f'{pos1:.2f} < pos < {pos2:.2f}' for pos1, pos2 in zip(pos_bins[:-1], pos_bins[1:])]
vel_labels = [f'{vel1:.2f} < vel < {vel2:.2f}' for vel1, vel2 in zip(vel_bins[:-1], vel_bins[1:])]


# Create MultiIndex for the Q-table
index = pd.MultiIndex.from_product([pos_labels, vel_labels], names=['pos', 'vel'])



columns = range(n_actions)

# Create Q-table with MultiIndex
q_table = pd.DataFrame(0, index=index, columns=columns)

In [ ]:
q_table

In [ ]:
pos_bins[1:-1]

In [ ]:
np.digitize([-1.3, -1.20, -1.1, 0.6, 5.0], pos_bins[1:-1])

In [ ]:
# Usually, we will create1a the multi-dim array in numpy

# Create a Q-table in numpy
q_table_np = np.arange(nbins_pos * nbins_vel * n_actions).reshape(nbins_pos, nbins_vel, n_actions)
print(q_table_np.shape)

In [ ]:
q_table_np

In [ ]:
def discretize_pos(pos):
    return np.digitize(pos, pos_bins[1:-1])

def discretize_vel(vel):
    return np.digitize(vel, vel_bins[1:-1])

from ipywidgets import interact

@interact(pos=(-1.3, 0.9, 0.01), vel=(-0.08, 0.09, 0.01))
def show_discretize(pos, vel):
    print(f"Position index {discretize_pos(pos)}, Velocity index {discretize_vel(vel)}, Value: {q_table_np[discretize_pos(pos), discretize_vel(vel)]}")


In [ ]:
# Use untrained Q-table to play the game

observation,  _  = env.reset(seed=42)

for i in range(100):
    env.render()
    pos, vel = observation
    pos_idx = discretize_pos(pos)
    vel_idx = discretize_vel(vel)
    action = np.argmax(q_table_np[pos_idx, vel_idx])
    observation, reward, terminated, truncated, info = env.step(action)
    print(i, observation, reward, terminated, truncated, info, action)
    if terminated:
        break
    

In [ ]:
# Train the Q-table

# Hyperparameters
alpha = 0.2 # learning rate
gamma = 0.99 # discount factor

# Exploration settings
epsilon = 0.8 # exploration rate

# Number of episodes
n_episodes = 4000

# Number of steps per episode
n_steps = 200

# Create a Q-table in numpy
nbins_pos = 25
nbins_vel = 25

pos_bins = np.linspace(env.observation_space.low[0], env.observation_space.high[0], nbins_pos)
vel_bins = np.linspace(env.observation_space.low[1], env.observation_space.high[1], nbins_vel)

pos_bins = np.concatenate([[-np.inf], pos_bins, [np.inf]])
vel_bins = np.concatenate([[-np.inf], vel_bins, [np.inf]])

q_table_np = np.random.randn(nbins_pos, nbins_vel, n_actions)

def discretize_pos(pos):
    return np.digitize(pos, pos_bins[1:-1])

def discretize_vel(vel):
    return np.digitize(vel, vel_bins[1:-1])
    

In [ ]:
# loop
# disable rendering
env = gym.make('MountainCar-v0', render_mode=None)
rewards = np.zeros(n_episodes)
for episode in range(n_episodes):
    epsilon = epsilon * 0.98
    if episode % 100 == 0:
        print(f"Episode {episode}")
    observation, _ = env.reset(seed=episode)
    cumulative_reward = 0
    for step in range(n_steps):
        # discretize the observation
        pos, vel = observation
        pos_idx = discretize_pos(pos)
        vel_idx = discretize_vel(vel)
        
        # select the action
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table_np[pos_idx, vel_idx])
        
        # take the action
        next_observation, reward, terminated, truncated, info = env.step(action)
        
        cumulative_reward += reward
        # discretize the next observation
        next_pos, next_vel = next_observation
        next_pos_idx = discretize_pos(next_pos)
        next_vel_idx = discretize_vel(next_vel)
        
        # update the q-table
        q_table_np[pos_idx, vel_idx, action] += alpha * (reward + gamma * np.max(q_table_np[next_pos_idx, next_vel_idx]) - q_table_np[pos_idx, vel_idx, action])
        
        observation = next_observation
        # rewards update
        rewards[episode] = cumulative_reward
        
        if terminated:
            print(f"Episode {episode} terminated after {step} steps")
            break

In [ ]:
q_table_np

In [ ]:
pd.Series(rewards).plot()

In [ ]:
# plot smoothed rewards
pd.Series(rewards).rolling(100).mean().plot()

In [ ]:
env = gym.make('MountainCar-v0', render_mode='human')
observation,  _  = env.reset(seed=42)

for i in range(300):
    env.render()
    pos, vel = observation
    pos_idx = discretize_pos(pos)
    vel_idx = discretize_vel(vel)
    action = np.argmax(q_table_np[pos_idx, vel_idx])
    observation, reward, terminated, truncated, info = env.step(action)
    print(i, observation, reward, terminated, truncated, info, action)
    if terminated:
        break

Reference: https://omarelb.github.io/dqn-investigation/

In [ ]:
# Deep Q-learning


class QNetwork(torch.nn.Module):
    def __init__(self, n_inputs, n_outputs, n_hidden=128):
        super(QNetwork, self).__init__()
        self.fc1 = torch.nn.Linear(n_inputs, n_hidden)
        self.fc2 = torch.nn.Linear(n_hidden, n_outputs)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
n_inputs = env.observation_space.shape[0]
n_outputs = env.action_space.n

q_network = QNetwork(n_inputs, n_outputs)

In [ ]:
q_network

In [ ]:
with torch.no_grad():
    q_values = q_network(torch.tensor([0.1, 0.2], dtype=torch.float32))
    print(q_values)


In [ ]:
# Train the Q-network

# Hyperparameters
alpha = 0.2 # learning rate
gamma = 0.99 # discount factor

# Exploration settings
epsilon = 0.2 # exploration rate
eps_mult = 0.99

# Number of episodes
n_episodes = 2000

# Number of steps per episode
n_steps = 300



In [ ]:
env = gym.make('MountainCar-v0', render_mode=None)

obs, _ = env.reset(seed=42)

q_network = QNetwork(n_inputs, n_outputs)

pos, vel = obs
pos_idx = discretize_pos(pos)
vel_idx = discretize_vel(vel)
print(observation)

with torch.no_grad():
    q_values = q_network(torch.tensor([pos, vel], dtype=torch.float32))
    print(q_values)

    action = torch.argmax(q_values).item()
    print(action)

    next_observation, reward, terminated, truncated, info = env.step(action)
    print(next_observation)

    next_pos, next_vel = next_observation

    next_pos_idx = discretize_pos(next_pos)
    next_vel_idx = discretize_vel(next_vel)

    next_q_values = q_network(torch.tensor([next_pos, next_vel], dtype=torch.float32))
    print(next_q_values)

    # Compute the target
    target = reward + gamma * torch.max(next_q_values)
    print(target)

# Compute the loss
criterion = torch.nn.MSELoss()
loss = criterion(q_values[action], target)
print(loss)




In [ ]:

optimizer = torch.optim.Adam(q_network.parameters(), lr=alpha)

for episode in range(n_episodes):
    epsilon = epsilon * eps_mult
    if episode % 100 == 0:
        print(f"Episode {episode}")
    observation, _ = env.reset(seed=1)
    cumulative_reward = 0
    for step in range(n_steps):
        # discretize the observation
        pos, vel = observation
        pos_idx = discretize_pos(pos)
        vel_idx = discretize_vel(vel)
        
        # select the action
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            q_values = q_network(torch.tensor([pos, vel], dtype=torch.float32))
            action = torch.argmax(q_values).item()
        
        # take the action
        next_observation, reward, terminated, truncated, info = env.step(action)
        
        cumulative_reward += reward
        # discretize the next observation
        next_pos, next_vel = next_observation
        next_pos_idx = discretize_pos(next_pos)
        next_vel_idx = discretize_vel(next_vel)
        
        # Compute Q-values for the next state
        with torch.no_grad():
            next_q_values = q_network(torch.tensor([next_pos, next_vel], dtype=torch.float32))
            max_next_q_value = torch.max(next_q_values).item()
        
        target_q_value = reward + gamma * max_next_q_value
        
        # Compute loss and update Q-network
        q_value = q_network(torch.tensor([pos, vel], dtype=torch.float32))[action]
        criterion = torch.nn.MSELoss()
        loss = criterion(q_value, torch.tensor(target_q_value, dtype=torch.float32))
        print(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        observation = next_observation
        # rewards update
        rewards[episode] = cumulative_reward

        
        if terminated:
            print(f"Episode {episode} terminated after {step} steps")
            break

In [ ]:
pd.Series(rewards).plot()

In [ ]:
import flappy_bird_gymnasium
env = gym.make("FlappyBird-v0", render_mode="human", use_lidar=False)

In [ ]:
obs, _ = env.reset()

In [ ]:
n_bins = 16
obs_low = env.observation_space.low
obs_high = env.observation_space.high

# Discretize the observation space
def discretize_observation(observation):
    bins = np.linspace(obs_low, obs_high, n_bins)
    return tuple(np.digitize(observation, bins))

# Define the variables for the MultiIndex
variables = [
    "last_pipe_h_pos",
    "last_top_pipe_v_pos",
    "last_bottom_pipe_v_pos",
    "next_pipe_h_pos",
    "next_top_pipe_v_pos",
    "next_bottom_pipe_v_pos",
    "next_next_pipe_h_pos",
    "next_next_top_pipe_v_pos",
    "next_next_bottom_pipe_v_pos",
    "player_v_pos",
    "player_v_vel",
    "player_rotation",
]

# leave out the first three and last three variables
var_consider = variables[3:-3]


In [ ]:
indices = [range(n_bins+1) for _ in var_consider] + [range(env.action_space.n)]

# Create a Q-table with a MultiIndex
q_table_np = np.random.randn(*[len(idx) for idx in indices]) * 0.2

q_table_np.shape

In [ ]:
obs_high, obs_low

In [ ]:
bins = np.linspace(obs_low[0], obs_high[0], n_bins)
bins

In [ ]:
def discretize_observation(observation):
    bins = np.linspace(obs_low[0], obs_high[0], n_bins)
    return tuple(np.digitize(observation, bins))

obs, _ = env.reset()

print(obs[3:-3], discretize_observation(obs[3:-3]))

In [ ]:
# Hyperparameters
alpha = 0.05 # learning rate

gamma = 0.99 # discount factor

# Exploration settings
epsilon = 1.0 # exploration rate

eps_mult = 0.99

# Number of episodes
n_episodes = 8000

# Number of steps per episode
n_steps = 500

In [ ]:
# q learning loop
env = gym.make("FlappyBird-v0", render_mode=None, use_lidar=False)
rewards = np.zeros(n_episodes)

for episode in range(n_episodes):
    epsilon = epsilon * eps_mult
    if episode % 100 == 0:
        print(f"Episode {episode}")
    observation, _ = env.reset(seed=episode)
    observation = observation[3:-3]
    cumulative_reward = 0
    for step in range(n_steps):
        # discretize the observation
        obs = discretize_observation(observation)
        
        # select the action
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table_np[obs])
        
        # take the action
        next_observation, reward, terminated, truncated, info = env.step(action)
        next_observation = next_observation[3:-3]
        cumulative_reward += reward
        # discretize the next observation
        next_obs = discretize_observation(next_observation)
        
        # update the q-table
        q_table_np[obs + (action,)] += alpha * (reward + gamma * np.max(q_table_np[next_obs]) - q_table_np[obs + (action,)])
        
        observation = next_observation
        # rewards update
        rewards[episode] = cumulative_reward
        
        if terminated:
            print(f"Episode {episode} terminated after {step} steps")
            break

In [ ]:
epsilon

In [ ]:
plt.plot(rewards)

plt.plot(pd.Series(rewards).rolling(100).mean())

In [ ]:
# now play in human mode using the trained Q-table

env = gym.make("FlappyBird-v0", render_mode="human", use_lidar=False)

obs, _ = env.reset()

for i in range(300):
    env.render()
    obs = obs[3:-3]
    obs_idx = discretize_observation(obs)
    action = np.argmax(q_table_np[obs_idx])
    action = int(i%16==0)
    obs, reward, terminated, truncated, info = env.step(action)
    print(i, action)
    if terminated:
        break

In [ ]:
# find mean for action


q_table_np[..., 0].mean(), q_table_np[..., 1].mean()